# LeetCode #1192: Critical Connections in a Network

https://leetcode.com/problems/critical-connections-in-a-network/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(E \cdot (V + E))$ | $O(V + E)$ |
| **Optimal: Tarjan's Bridge Finding ★** | $O(V + E)$ | $O(V + E)$ |

---

## Understanding the Methods

### Brute Force
Remove each edge and check if the graph remains connected (BFS/DFS). An edge that disconnects the graph is a bridge. Requires $O(E)$ removals each costing $O(V + E)$.

### Optimal: Tarjan's Bridge Finding ★
One DFS pass assigns each node a discovery time. The `low` value for a node is the earliest discovery time reachable via the DFS subtree (including back edges). An edge $(u, v)$ is a bridge if and only if $low[v] > disc[u]$ — meaning the subtree rooted at $v$ cannot reach $u$ or any ancestor of $u$ without passing through this edge.

**Constraints:**
* $2 \le n \le 10^5$
* $n - 1 \le connections.length \le 10^5$

## Solutions

### C#

In [ ]:
using System.Collections.Generic;

public class Solution {
    int[] disc, low;
    bool[] visited;
    List<int>[] adj;
    List<IList<int>> bridges;
    int timer;

    public IList<IList<int>> CriticalConnections(int n, IList<IList<int>> connections) {
        disc = new int[n]; low = new int[n]; visited = new bool[n];
        adj = new List<int>[n];
        for (int i = 0; i < n; i++) adj[i] = new List<int>();
        bridges = new List<IList<int>>();

        foreach (var c in connections) {
            adj[c[0]].Add(c[1]);
            adj[c[1]].Add(c[0]);
        }

        for (int i = 0; i < n; i++)
            if (!visited[i]) Dfs(i, -1);

        return bridges;
    }

    void Dfs(int u, int parent) {
        visited[u] = true;
        disc[u] = low[u] = timer++;

        foreach (int v in adj[u]) {
            if (!visited[v]) {
                Dfs(v, u);
                // Pull up the lowest reachable time from the child subtree
                low[u] = Math.Min(low[u], low[v]);
                // If v can't reach u or higher — this edge is a bridge
                if (low[v] > disc[u])
                    bridges.Add(new List<int>{u, v});
            } else if (v != parent) {
                // Back edge — update low with the already-visited neighbour's disc time
                low[u] = Math.Min(low[u], disc[v]);
            }
        }
    }
}

### Python

In [ ]:
from collections import defaultdict

class Solution:
    def criticalConnections(self, n: int, connections: list[list[int]]) -> list[list[int]]:
        adj = defaultdict(list)
        for u, v in connections:
            adj[u].append(v)
            adj[v].append(u)

        disc = [-1] * n
        low = [0] * n
        bridges = []
        timer = [0]

        def dfs(u: int, parent: int) -> None:
            disc[u] = low[u] = timer[0]
            timer[0] += 1
            for v in adj[u]:
                if disc[v] == -1:
                    dfs(v, u)
                    # Propagate the lowest reachable time up from child
                    low[u] = min(low[u], low[v])
                    # Child's subtree can't reach u or above — bridge found
                    if low[v] > disc[u]:
                        bridges.append([u, v])
                elif v != parent:
                    # Back edge — can skip the parent to avoid false cycles
                    low[u] = min(low[u], disc[v])

        for i in range(n):
            if disc[i] == -1:
                dfs(i, -1)
        return bridges

### Go

In [ ]:
func criticalConnections(n int, connections [][]int) [][]int {
	adj := make([][]int, n)
	for _, c := range connections {
		u, v := c[0], c[1]
		adj[u] = append(adj[u], v)
		adj[v] = append(adj[v], u)
	}

	disc := make([]int, n)
	low := make([]int, n)
	for i := range disc { disc[i] = -1 }
	timer := 0
	var bridges [][]int

	var dfs func(u, parent int)
	dfs = func(u, parent int) {
		disc[u] = timer; low[u] = timer; timer++
		for _, v := range adj[u] {
			if disc[v] == -1 {
				dfs(v, u)
				// Bring up the lowest reachable time from the child
				if low[v] < low[u] { low[u] = low[v] }
				// No back-edge from v's subtree to u or above — bridge
				if low[v] > disc[u] {
					bridges = append(bridges, []int{u, v})
				}
			} else if v != parent {
				// Back edge to an already-visited ancestor
				if disc[v] < low[u] { low[u] = disc[v] }
			}
		}
	}

	for i := 0; i < n; i++ {
		if disc[i] == -1 { dfs(i, -1) }
	}
	return bridges
}

### Rust

In [ ]:
impl Solution {
    pub fn critical_connections(n: i32, connections: Vec<Vec<i32>>) -> Vec<Vec<i32>> {
        let n = n as usize;
        let mut adj = vec![vec![]; n];
        for c in &connections {
            adj[c[0] as usize].push(c[1] as usize);
            adj[c[1] as usize].push(c[0] as usize);
        }

        let mut disc = vec![usize::MAX; n];
        let mut low = vec![0usize; n];
        let mut timer = 0usize;
        let mut bridges = Vec::new();

        // Iterative Tarjan using an explicit stack to avoid stack overflow
        let mut stack: Vec<(usize, usize, usize)> = Vec::new(); // (node, parent, adj_idx)
        for start in 0..n {
            if disc[start] != usize::MAX { continue; }
            stack.push((start, usize::MAX, 0));
            disc[start] = timer; low[start] = timer; timer += 1;

            while let Some((u, par, idx)) = stack.last_mut() {
                let u = *u; let par = *par;
                if *idx < adj[u].len() {
                    let v = adj[u][*idx]; *idx += 1;
                    if disc[v] == usize::MAX {
                        disc[v] = timer; low[v] = timer; timer += 1;
                        stack.push((v, u, 0));
                    } else if v != par {
                        low[u] = low[u].min(disc[v]);
                    }
                } else {
                    stack.pop();
                    if let Some(&mut (pu, _, _)) = stack.last_mut().as_mut() {
                        let pu = pu;
                        if low[u] < low[pu] { low[pu] = low[u]; }
                        if low[u] > disc[pu] {
                            bridges.push(vec![pu as i32, u as i32]);
                        }
                    }
                }
            }
        }
        bridges
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `n=4, connections=[[0,1],[1,2],[2,0],[1,3]]`
Nodes 0-1-2 form a cycle so no bridge exists among them. Edge (1,3) is the only bridge because removing it disconnects node 3. Answer: `[[1,3]]`.

### 2. Slightly Complex
**Input:** `n=5, connections=[[0,1],[0,2],[1,2],[2,3],[3,4]]`
0-1-2 form a cycle (no bridges). (2,3) and (3,4) are bridges because nodes 3 and 4 have no alternate paths. Answer: `[[2,3],[3,4]]`.

### 3. Edge Case: Time Factor
**Input:** A path graph: $n=10^5$, connections form a single chain $0-1-2-\cdots-(n-1)$.
Every edge is a bridge. Tarjan's DFS visits each of the $n-1$ edges exactly once, running in $O(n)$.

### 4. Edge Case: Space Factor
**Input:** A complete graph $K_{1000}$ (every node connected to every other).
Adjacency list has $O(n^2)$ entries; $disc$ and $low$ arrays are $O(n)$; the DFS stack is $O(n)$ deep. No bridges exist because every edge lies on a cycle.

### 5. Almost-Impossible but Plausible
**Input:** Two large cliques connected by a single edge.
The single inter-clique edge is the only bridge. Tarjan discovers it in $O(V + E)$ regardless of clique size, while the brute-force approach would require $O(E \cdot (V + E))$ time.